In [1]:
import h5py
import numpy as np
import pandas as pd
import joblib
import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Input, Dense, LSTM, BatchNormalization, Dropout, Flatten, SimpleRNN, GRU
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, History
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from sklearn.preprocessing import LabelEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

**Wczytanie danych treningowych**

In [2]:
with h5py.File('/kaggle/input/mfcc-data-for-ubm/data.h5', 'r') as f:
    X_train = f['X_train'][:]
    y_train = f['y_train'][:]
    X_val = f['X_val'][:]
    y_val = f['y_val'][:]
    X_train_delta = f['X_train_delta'][:]
    X_val_delta = f['X_val_delta'][:]
    X_emb_delta = f['X_emb_delta'][:]

In [3]:
input_shape = (X_train.shape[1], X_train.shape[2])

**Definicja i trening modelu UBM**

Do porównania wybrano modele o architekturze RNN z zastosowaniem technik Simple RNN, GRU oraz LSTM. Model, który uzyska najlepszą wydajność, zostanie ponownie wytrenowany, tym razem na danych MFCC wzbogaconych o pochodne pierwszego i drugiego rzędu (delta oraz delta-delta). Dodatkowo, po treningu zastosowana zostanie analiza dyskryminacyjna liniowa (LDA) w celu postprocessingu embeddingów.

In [4]:
Simple_RNN = Sequential()
Simple_RNN.add(Input(shape=input_shape))

# Bloki RNN
Simple_RNN.add(SimpleRNN(64, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.3))

Simple_RNN.add(SimpleRNN(128, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.3))

Simple_RNN.add(SimpleRNN(256, return_sequences=False, activation='tanh', kernel_regularizer=l2(0.0005)))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.4))

Simple_RNN.add(Flatten())

# Warstwa bottleneck
Simple_RNN.add(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.2))

Simple_RNN.add(Dense(120, activation='relu', kernel_regularizer=l2(0.001), name="bottleneck"))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.1))

Simple_RNN.add(Dense(150, activation='relu', kernel_regularizer=l2(0.001)))
Simple_RNN.add(BatchNormalization())
Simple_RNN.add(Dropout(0.1))
Simple_RNN.add(Dense(100, activation='softmax'))

Simple_RNN.compile(optimizer=Adam(learning_rate=0.0001), 
            loss='categorical_crossentropy', 
            metrics=['Precision'])

Simple_RNN.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ simple_rnn (SimpleRNN)          │ (None, 16, 64)         │         4,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 16, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (None, 256)            │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 120)            │           480 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 150)            │        18,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 150)            │           600 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        15,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 262,034 (1023.57 KB)

 Trainable params: 260,086 (1015.96 KB)

 Non-trainable params: 1,948 (7.61 KB)

In [5]:
lstm = Sequential()
lstm.add(Input(shape=input_shape))

# Bloki RNN
lstm.add(LSTM(64, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.3))

lstm.add(LSTM(128, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.3))

lstm.add(LSTM(256, return_sequences=False, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.3))

lstm.add(Flatten())

# Warstwa bottleneck
lstm.add(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.2))

lstm.add(Dense(120, activation='relu', kernel_regularizer=l2(0.001), name="bottleneck"))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.1))

lstm.add(Dense(150, activation='relu', kernel_regularizer=l2(0.001)))
lstm.add(BatchNormalization())
lstm.add(Dropout(0.1))
lstm.add(Dense(100, activation='softmax'))

lstm.compile(optimizer=Adam(learning_rate=0.0001), 
            loss='categorical_crossentropy', 
            metrics=['Precision'])

lstm.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 16, 64)         │        19,968 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_6           │ (None, 16, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 16, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_7           │ (None, 16, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_10          │ (None, 120)            │           480 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 150)            │        18,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_11          │ (None, 150)            │           600 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 100)            │        15,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 646,802 (2.47 MB)

 Trainable params: 644,854 (2.46 MB)

 Non-trainable params: 1,948 (7.61 KB)

In [6]:
gru = Sequential()
gru.add(Input(shape=input_shape))

# Bloki RNN
gru.add(GRU(64, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
gru.add(BatchNormalization())
gru.add(Dropout(0.3))

gru.add(GRU(128, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
gru.add(BatchNormalization())
gru.add(Dropout(0.3))

gru.add(GRU(256, return_sequences=False, activation='tanh', kernel_regularizer=l2(0.0005)))
gru.add(BatchNormalization())
gru.add(Dropout(0.3))

gru.add(Flatten())

# Warstwa bottleneck
gru.add(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))
gru.add(BatchNormalization())
gru.add(Dropout(0.2))

gru.add(Dense(120, activation='relu', kernel_regularizer=l2(0.001), name="bottleneck"))
gru.add(BatchNormalization())
gru.add(Dropout(0.1))

gru.add(Dense(150, activation='relu', kernel_regularizer=l2(0.001)))
gru.add(BatchNormalization())
gru.add(Dropout(0.1))
gru.add(Dense(100, activation='softmax'))

gru.compile(optimizer=Adam(learning_rate=0.001), 
            loss='categorical_crossentropy', 
            metrics=['Precision'])

gru.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 16, 64)         │        15,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 16, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 16, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ (None, 16, 128)        │        74,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 16, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_13 (Dropout)            │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_2 (GRU)                     │ (None, 256)            │       296,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_14 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_2 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_15 (Dropout)            │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_16          │ (None, 120)            │           480 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_16 (Dropout)            │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 150)            │        18,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_17          │ (None, 150)            │           600 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_17 (Dropout)            │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100)            │        15,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 519,890 (1.98 MB)

 Trainable params: 517,942 (1.98 MB)

 Non-trainable params: 1,948 (7.61 KB)

**Trening modeli**

W celu zoptymalizowania procesu treningu zastosowano techniki wczesnego zatrzymania oraz redukcji tempa uczenia. 

In [7]:
early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=30,         
    restore_best_weights=True  
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.000001       
)

epochs = 150

history_SRNN = Simple_RNN.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=epochs,  
                    batch_size=32,  
                    verbose=1,
                    callbacks=[reduce_lr])  


Simple_RNN.save("SRNN.h5")

Epoch 1/150


I0000 00:00:1732394484.393959      77 service.cc:145] XLA service 0x55f91316c9a0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1732394484.394008      77 service.cc:153]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1732394484.394012      77 service.cc:153]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5


  18/3512 ━━━━━━━━━━━━━━━━━━━━ 34s 10ms/step - Precision: 0.0000e+00 - loss: 5.9472  

I0000 00:00:1732394501.552490      77 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


3512/3512 ━━━━━━━━━━━━━━━━━━━━ 66s 12ms/step - Precision: 0.0747 - loss: 5.5437 - val_Precision: 0.6437 - val_loss: 4.0556 - learning_rate: 1.0000e-04
Epoch 2/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - Precision: 0.6641 - loss: 4.1773 - val_Precision: 0.7467 - val_loss: 3.3780 - learning_rate: 1.0000e-04
Epoch 3/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - Precision: 0.5819 - loss: 3.6219 - val_Precision: 0.7008 - val_loss: 2.9461 - learning_rate: 1.0000e-04
Epoch 4/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - Precision: 0.6148 - loss: 3.2701 - val_Precision: 0.6780 - val_loss: 2.6997 - learning_rate: 1.0000e-04
Epoch 5/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - Precision: 0.6262 - loss: 3.0425 - val_Precision: 0.7012 - val_loss: 2.4714 - learning_rate: 1.0000e-04
Epoch 6/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - Precision: 0.6383 - loss: 2.8511 - val_Precision: 0.7167 - val_loss: 2.2588 - learning_rate: 1.0000e-04
Epoch 7/150
3512/3512 ━━━━━━━━━━━━━━━━━

In [8]:
history_lstm = lstm.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=epochs,  
                    batch_size=32,  
                    verbose=1,
                    callbacks=[reduce_lr])  

lstm.save("lstm.h5")

Epoch 1/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 61s 15ms/step - Precision: 0.4753 - loss: 5.0470 - val_Precision: 0.7821 - val_loss: 3.0953 - learning_rate: 1.0000e-04
Epoch 2/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 53s 15ms/step - Precision: 0.7033 - loss: 3.3425 - val_Precision: 0.8092 - val_loss: 2.2144 - learning_rate: 1.0000e-04
Epoch 3/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 53s 15ms/step - Precision: 0.7426 - loss: 2.6375 - val_Precision: 0.8322 - val_loss: 1.7443 - learning_rate: 1.0000e-04
Epoch 4/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 53s 15ms/step - Precision: 0.7637 - loss: 2.2269 - val_Precision: 0.8480 - val_loss: 1.4610 - learning_rate: 1.0000e-04
Epoch 5/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 53s 15ms/step - Precision: 0.7752 - loss: 1.9605 - val_Precision: 0.8621 - val_loss: 1.2689 - learning_rate: 1.0000e-04
Epoch 6/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 53s 15ms/step - Precision: 0.7950 - loss: 1.7457 - val_Precision: 0.8721 - val_loss: 1.1292 - learning_rate: 1.0000e-04
Epoch 7/150
3512/3512 

In [9]:
history_gru = gru.fit(X_train, y_train,
                    validation_data=(X_val, y_val),
                    epochs=epochs,  
                    batch_size=32,  
                    verbose=1,
                    callbacks=[ reduce_lr])  

gru.save('gru.h5')

Epoch 1/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 58s 15ms/step - Precision: 0.5914 - loss: 3.6408 - val_Precision: 0.7686 - val_loss: 1.7946 - learning_rate: 0.0010
Epoch 2/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - Precision: 0.7232 - loss: 2.0596 - val_Precision: 0.8153 - val_loss: 1.4104 - learning_rate: 0.0010
Epoch 3/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - Precision: 0.7540 - loss: 1.7738 - val_Precision: 0.8234 - val_loss: 1.2805 - learning_rate: 0.0010
Epoch 4/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - Precision: 0.7748 - loss: 1.6371 - val_Precision: 0.8423 - val_loss: 1.2319 - learning_rate: 0.0010
Epoch 5/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - Precision: 0.7881 - loss: 1.5653 - val_Precision: 0.8519 - val_loss: 1.1693 - learning_rate: 0.0010
Epoch 6/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s 15ms/step - Precision: 0.7993 - loss: 1.4744 - val_Precision: 0.8542 - val_loss: 1.1260 - learning_rate: 0.0010
Epoch 7/150
3512/3512 ━━━━━━━━━━━━━━━━━━━━ 52s

Najlepszą architekturą okazała się architektura oparta na LSTM, dlatego zostanie ona ponownie wytrenowana na danych MFCC wzbogaconych o pochodne (delta).

In [4]:
input_shape_delta =  (X_train_delta.shape[1], X_train_delta.shape[2])

lstm_delta = Sequential()
lstm_delta.add(Input(shape=input_shape_delta))

# Bloki RNN
lstm_delta.add(LSTM(64, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.3))

lstm_delta.add(LSTM(128, return_sequences=True, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.3))

lstm_delta.add(LSTM(256, return_sequences=False, activation='tanh', kernel_regularizer=l2(0.0005)))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.3))

lstm_delta.add(Flatten())

# Warstwa bottleneck
lstm_delta.add(Dense(256, activation='relu', kernel_regularizer=l2(0.001)))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.2))

lstm_delta.add(Dense(120, activation='relu', kernel_regularizer=l2(0.001), name="bottleneck"))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.1))

lstm_delta.add(Dense(150, activation='relu', kernel_regularizer=l2(0.001)))
lstm_delta.add(BatchNormalization())
lstm_delta.add(Dropout(0.1))
lstm_delta.add(Dense(100, activation='softmax'))

lstm_delta.compile(optimizer=Adam(learning_rate=0.0001), 
            loss='categorical_crossentropy', 
            metrics=['Precision'])

lstm_delta.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 16, 64)         │        26,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 16, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 16, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 16, 128)        │        98,816 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 16, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 16, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 256)            │       394,240 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bottleneck (Dense)              │ (None, 120)            │        30,840 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 120)            │           480 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ (None, 120)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 150)            │        18,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 150)            │           600 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 150)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        15,100 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 653,458 (2.49 MB)

 Trainable params: 651,510 (2.49 MB)

 Non-trainable params: 1,948 (7.61 KB)

In [5]:
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.000001       
)


history_lstm = lstm_delta.fit(X_train_delta, y_train,
                    validation_data=(X_val_delta, y_val),
                    epochs=200,  
                    batch_size=64,  
                    verbose=1)  

lstm_delta.save('lstm_delta.h5')

Epoch 1/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 32s 14ms/step - Precision: 0.4543 - loss: 5.2348 - val_Precision: 0.7829 - val_loss: 3.5188
Epoch 2/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.6883 - loss: 3.6978 - val_Precision: 0.8098 - val_loss: 2.7383
Epoch 3/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.7286 - loss: 3.0390 - val_Precision: 0.8179 - val_loss: 2.2156
Epoch 4/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.7516 - loss: 2.5875 - val_Precision: 0.8215 - val_loss: 1.8763
Epoch 5/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.7658 - loss: 2.2657 - val_Precision: 0.8408 - val_loss: 1.6220
Epoch 6/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.7767 - loss: 2.0229 - val_Precision: 0.8404 - val_loss: 1.4653
Epoch 7/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/step - Precision: 0.7847 - loss: 1.8491 - val_Precision: 0.8517 - val_loss: 1.3251
Epoch 8/200
1756/1756 ━━━━━━━━━━━━━━━━━━━━ 24s 14ms/ste

Model wytrenowany na wzbogaconych danych charakteryzuje się mniejszą wartością funkcji straty.

**LDA**

Aby uniknąć przecieku danych i zawyżenia wyników klasyfikacji, model LDA musi być trenowany wyłącznie na danych treningowych. Oznacza to, że transformacja LDA, która redukuje wymiarowość i optymalizuje rozdzielczość między klasami, jest dostosowywana wyłącznie do struktury danych treningowych.

In [13]:
intermediate_layer_model = Model(inputs=lstm_delta.layers[0].input,
                                 outputs=lstm_delta.get_layer("bottleneck").output)
intermediate_output = intermediate_layer_model.predict(X_train_delta)

train_embeddings = np.array(intermediate_output)
y_labels = np.argmax(y_train, axis = 1)

3512/3512 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step


Wybór liczby komponentów n_components dla modelu LDA opiera się na dwóch ograniczeniach: liczbie klas w danych oraz liczbie dostępnych wymiarów w danych wejściowych. Wybrano minimalną wartość spośród tych dwóch ograniczeń.

In [14]:
n_components = min(len(set(y_labels)) - 1, train_embeddings.shape[1])
lda = LDA(n_components=n_components)
lda.fit(train_embeddings, y_labels)

joblib.dump(lda, 'lda.joblib')

['lda.joblib']